In [1]:
pip install yfinance

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [2]:
pip install backtesting

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
import yfinance as yf
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

# 1. 抓取台積電 (2330.TW) 歷史資料 (2000 ~ 2020)
print("正在從 Yahoo Finance 獲取台積電歷史數據...")
df = yf.download('2317.TW', start='2000-01-01', end='2025-12-31')

# 處理 yfinance 新版本可能帶有的多層欄位索引 (MultiIndex)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]
df = df.dropna()

# 2. 計算技術指標
# 條件1: 單日漲跌幅(%)
df['Price_Change_Pct'] = df['Close'].pct_change() * 100

# 條件3: 20日均線與乖離率(%)
df['MA20'] = df['Close'].rolling(window=20).mean()
df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100

# 條件2 & 4: KD值 (參數採用台灣市場常用的 9, 3, 3)
df['Min9'] = df['Low'].rolling(window=9).min()
df['Max9'] = df['High'].rolling(window=9).max()
df['RSV'] = (df['Close'] - df['Min9']) / (df['Max9'] - df['Min9']) * 100
df['RSV'] = df['RSV'].fillna(50)

K, D = [50], [50]
for rsv in df['RSV'].values[1:]:
    k_val = K[-1] * (2/3) + rsv * (1/3)
    d_val = D[-1] * (2/3) + k_val * (1/3)
    K.append(k_val)
    D.append(d_val)
df['K'] = K
df['D'] = D

# 特定條件1: 日MACD落點 (DIF, MACD, OSC 皆大於0)
df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean() # 訊號線
df['OSC'] = df['DIF'] - df['MACD']                      # 柱狀體

# 3. 建立進場布林條件陣列 (完全遵循您的自訂篩選面板)
c1 = df['Price_Change_Pct'].between(5, 8)
c2 = df['K'].between(30, 70)
c3 = df['Bias20'].between(5, 7)
c4 = df['D'].between(30, 70)
c5 = (df['DIF'] > 0) & (df['MACD'] > 0) & (df['OSC'] > 0)
c6 = df['Close'] > df['MA20']

# 綜合進場總訊號
df['Buy_Signal'] = c1 & c2 & c3 & c4 & c5 & c6

# 4. 修改後的進出場狀態機邏輯 (進場嚴格，出場看均線與趨勢)
trades = []
holding = False
buy_date = None
buy_price = 0.0

for date, row in df.iterrows():
    entry_signal = row['Buy_Signal']
    
    # 【核心修改點】出場條件：當收盤價跌破20日均線，或者 MACD 快線(DIF)轉為小於等於0時賣出
    exit_signal = (row['Close'] < row['MA20']) or (row['DIF'] <= 0)
    
    if not holding and entry_signal:
        # 觸發買入點
        holding = True
        buy_date = date
        buy_price = row['Close']
        
    elif holding and exit_signal:
        # 觸發出場點
        holding = False
        sell_date = date
        sell_price = row['Close']
        return_pct = (sell_price - buy_price) / buy_price
        
        trades.append({
            'Buy Date': buy_date.strftime('%Y-%m-%d'),
            'Buy Price': round(buy_price, 2),
            'Sell Date': sell_date.strftime('%Y-%m-%d'),
            'Sell Price': round(sell_price, 2),
            'Return (%)': round(return_pct, 4),
            'Holding Days': (sell_date - buy_date).days
        })

# 請將這段代碼接在計算完 c1 ~ c6 條件之後執行
print("===== 策略條件留存率診斷 =====")
print(f"總交易天數 (2000-2020): {len(df)} 天")
print(f"1. 符合 漲跌幅 5%~8% 的天數: {c1.sum()} 天")
print(f"2. 符合 K值 30~70 的天數: {c2.sum()} 天")
print(f"3. 符合 20日乖離 5%~7% 的天數: {c3.sum()} 天")
print(f"4. 符合 D值 30~70 的天數: {c4.sum()} 天")
print(f"5. 符合 MACD 三指標皆 > 0 的天數: {c5.sum()} 天")
print(f"6. 符合 股價在MA20之上的天數: {c6.sum()} 天")
print("--------------------------------")
print(f"最後六個條件同時交集（進場訊號）的天數: {df['Buy_Signal'].sum()} 天")

# 處理邊界狀況：若回測最後一天仍持有，則強制進行結算統計
if holding:
    last_date = df.index[-1]
    last_row = df.iloc[-1]
    sell_price = last_row['Close']
    return_pct = (sell_price - buy_price) / buy_price
    trades.append({
        'Buy Date': buy_date.strftime('%Y-%m-%d'),
        'Buy Price': round(buy_price, 2),
        'Sell Date': last_date.strftime('%Y-%m-%d') + " (未平倉強制結算)",
        'Sell Price': round(sell_price, 2),
        'Return (%)': round(return_pct, 4),
        'Holding Days': (last_date - buy_date).days
    })

# 轉為 DataFrame
trades_df = pd.DataFrame(trades)
if trades_df.empty:
    print("提示：在 2000-2020 期間，此嚴格選股條件沒有觸發任何交易。")
    trades_df = pd.DataFrame(columns=['Buy Date', 'Buy Price', 'Sell Date', 'Sell Price', 'Return (%)', 'Holding Days'])
else:
    print(f"優化策略成功！總共執行了 {len(trades)} 筆交易。")

# 5. 匯出至專業排版 Excel
output_path = 'TSMC_Optimized_Backtest.xlsx'
wb = Workbook()
ws_summary = wb.active
ws_summary.title = "優化策略交易紀錄"

# 寫入欄位與數據
for r in dataframe_to_rows(trades_df, index=False, header=True):
    ws_summary.append(r)

# 樣式與色彩美化設定 (商務深藍風)
header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
center_align = Alignment(horizontal="center", vertical="center")
border_style = Border(left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'), 
                      top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9'))

# 標題列套用樣式
for cell in ws_summary[1]:
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = center_align
    cell.border = border_style

# 資料列套用樣式與格式化
for row in ws_summary.iter_rows(min_row=2, max_row=ws_summary.max_row, min_col=1, max_col=6):
    for cell in row:
        cell.border = border_style
        cell.alignment = center_align
        if cell.column == 5:  # Return (%) 欄位設定為百分比格式
            cell.number_format = '0.00%'

# 自動根據內容寬度調整 Excel 欄寬
for col in ws_summary.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    col_letter = col[0].column_letter
    ws_summary.column_dimensions[col_letter].width = max(max_len + 3, 12)


  
wb.save(output_path)
print(f"Excel 報表已儲存至：{output_path}")

正在從 Yahoo Finance 獲取台積電歷史數據...


[*********************100%***********************]  1 of 1 completed


===== 策略條件留存率診斷 =====
總交易天數 (2000-2020): 6469 天
1. 符合 漲跌幅 5%~8% 的天數: 126 天
2. 符合 K值 30~70 的天數: 3054 天
3. 符合 20日乖離 5%~7% 的天數: 401 天
4. 符合 D值 30~70 的天數: 3674 天
5. 符合 MACD 三指標皆 > 0 的天數: 1851 天
6. 符合 股價在MA20之上的天數: 3596 天
--------------------------------
最後六個條件同時交集（進場訊號）的天數: 4 天
優化策略成功！總共執行了 4 筆交易。
Excel 報表已儲存至：TSMC_Optimized_Backtest.xlsx


# 原始價格

In [34]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# 1. 獲取資料 (鎖定原始收盤價)
# ==========================================
print("正在獲取台積電(2330.TW)歷史資料...")
df = yf.download('2330.TW', start='2000-01-01', end='2020-12-31', auto_adjust=False, progress=False)

# 處理欄位名稱 (避免新版 yfinance 的 MultiIndex 問題)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

# 剔除還原收盤價，確保完全使用原始價格
if 'Adj Close' in df.columns:
    df = df.drop(columns=['Adj Close'])
df = df.dropna()

# ==========================================
# 2. 計算技術指標
# ==========================================
# 漲跌幅(%)
df['Price_Change_Pct'] = df['Close'].pct_change() * 100

# 20日均線與乖離率(%)
df['MA20'] = df['Close'].rolling(window=20).mean()
df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100

# KD值 (9, 3, 3)
df['Min9'] = df['Low'].rolling(window=9).min()
df['Max9'] = df['High'].rolling(window=9).max()
df['RSV'] = (df['Close'] - df['Min9']) / (df['Max9'] - df['Min9']) * 100
df['RSV'] = df['RSV'].fillna(50)

K, D = [50], [50]
for rsv in df['RSV'].values[1:]:
    k_val = K[-1] * (2/3) + rsv * (1/3)
    d_val = D[-1] * (2/3) + k_val * (1/3)
    K.append(k_val)
    D.append(d_val)
df['K'] = K
df['D'] = D

# MACD (12, 26, 9)
df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean() 
df['OSC'] = df['DIF'] - df['MACD']                      

# ==========================================
# 3. 建立進出場訊號
# ==========================================
# 進場條件 (嚴格交集)
c1 = df['Price_Change_Pct'].between(5, 8)
c2 = df['K'].between(30, 70)
c3 = df['Bias20'].between(5, 7)
c4 = df['D'].between(30, 70)
c5 = (df['DIF'] > 0) & (df['MACD'] > 0) & (df['OSC'] > 0)
c6 = df['Close'] > df['MA20']
df['Buy_Signal'] = c1 & c2 & c3 & c4 & c5 & c6

# ==========================================
# 4. 回測狀態機
# ==========================================
trades = []
holding = False
buy_date, buy_price = None, 0.0

for date, row in df.iterrows():
    entry_signal = row['Buy_Signal']
    # 出場：跌破20日均線 或 MACD動能轉弱(DIF<=0)
    exit_signal = (row['Close'] < row['MA20']) or (row['DIF'] <= 0)
    
    if not holding and entry_signal:
        holding = True
        buy_date = date
        buy_price = row['Close']
        
    elif holding and exit_signal:
        holding = False
        sell_date = date
        sell_price = row['Close']
        return_pct = (sell_price - buy_price) / buy_price
        
        trades.append({
            'Buy_Date': buy_date.strftime('%Y-%m-%d'),
            'Buy_Price': round(buy_price, 2),
            'Sell_Date': sell_date.strftime('%Y-%m-%d'),
            'Sell_Price': round(sell_price, 2),
            'Return_Pct': round(return_pct * 100, 2), # 轉為百分比顯示
            'Holding_Days': (sell_date - buy_date).days
        })

# 處理最後一天未平倉狀況
if holding:
    last_date = df.index[-1]
    sell_price = df.iloc[-1]['Close']
    return_pct = (sell_price - buy_price) / buy_price
    trades.append({
        'Buy_Date': buy_date.strftime('%Y-%m-%d'),
        'Buy_Price': round(buy_price, 2),
        'Sell_Date': last_date.strftime('%Y-%m-%d') + " (未平倉)",
        'Sell_Price': round(sell_price, 2),
        'Return_Pct': round(return_pct * 100, 2),
        'Holding_Days': (last_date - buy_date).days
    })

# ==========================================
# 5. 輸出 DataFrame 與績效總結
# ==========================================
trades_df = pd.DataFrame(trades)

print("\n" + "="*40)
if trades_df.empty:
    print("在此期間內，沒有任何一天同時滿足所有嚴格進場條件。")
else:
    # 計算簡易績效指標
    total_trades = len(trades_df)
    win_trades = len(trades_df[trades_df['Return_Pct'] > 0])
    win_rate = (win_trades / total_trades) * 100
    avg_return = trades_df['Return_Pct'].mean()
    
    print(f"回測完成！總交易次數: {total_trades} 次")
    print(f"策略勝率: {win_rate:.2f}%")
    print(f"單筆平均報酬率: {avg_return:.2f}%")
    print("="*40 + "\n")
    
    # 若在一般 IDE (如 VS Code, PyCharm) 執行，使用 print 顯示前幾筆或全部
    # print(trades_df.to_string()) 
    
# 如果在 Jupyter Notebook 中，請在下一個儲存格直接輸入 trades_df 即可顯示表格
display(trades_df)

正在獲取台積電(2330.TW)歷史資料...

回測完成！總交易次數: 3 次
策略勝率: 33.33%
單筆平均報酬率: -0.81%



,Buy_Date,Buy_Price,Sell_Date,Sell_Price,Return_Pct,Holding_Days
0,2000-04-10,82.17,2000-04-12,78.26,-4.76,2
1,2009-03-23,50.05,2009-04-08,49.95,-0.20,16
2,2009-04-30,54.93,2009-06-08,56.32,2.54,39


# 20%出場條件

In [37]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# 1. 獲取資料 (鎖定原始收盤價)
# ==========================================
print("正在獲取台積電(2330.TW)歷史資料...")
df = yf.download('2330.TW', start='2000-01-01', end='2020-12-31', auto_adjust=False, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

if 'Adj Close' in df.columns:
    df = df.drop(columns=['Adj Close'])
df = df.dropna()

# ==========================================
# 2. 計算技術指標 (全數基於原始收盤價)
# ==========================================
df['Price_Change_Pct'] = df['Close'].pct_change() * 100

df['MA20'] = df['Close'].rolling(window=20).mean()
df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100

df['Min9'] = df['Low'].rolling(window=9).min()
df['Max9'] = df['High'].rolling(window=9).max()
df['RSV'] = (df['Close'] - df['Min9']) / (df['Max9'] - df['Min9']) * 100
df['RSV'] = df['RSV'].fillna(50)

K, D = [50], [50]
for rsv in df['RSV'].values[1:]:
    k_val = K[-1] * (2/3) + rsv * (1/3)
    d_val = D[-1] * (2/3) + k_val * (1/3)
    K.append(k_val)
    D.append(d_val)
df['K'] = K
df['D'] = D

df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean() 
df['OSC'] = df['DIF'] - df['MACD']                      

# ==========================================
# 3. 建立進場訊號 (維持您的嚴格選股條件)
# ==========================================
c1 = df['Price_Change_Pct'].between(5, 8)
c2 = df['K'].between(30, 70)
c3 = df['Bias20'].between(5, 7)
c4 = df['D'].between(30, 70)
c5 = (df['DIF'] > 0) & (df['MACD'] > 0) & (df['OSC'] > 0)
c6 = df['Close'] > df['MA20']
df['Buy_Signal'] = c1 & c2 & c3 & c4 & c5 & c6

# ==========================================
# 4. 回測狀態機 (【修改區域】: 固定 20% 停利停損)
# ==========================================
trades = []
holding = False
buy_date, buy_price = None, 0.0

for date, row in df.iterrows():
    entry_signal = row['Buy_Signal']
    
    if not holding and entry_signal:
        holding = True
        buy_date = date
        buy_price = row['Close']
        
    elif holding:
        # 計算當前(今日收盤)相較於買入價的未實現報酬率
        current_price = row['Close']
        unrealized_return = (current_price - buy_price) / buy_price
        
        # 出場條件：獲利 >= 20% 或 虧損 <= -20%
        if unrealized_return >= 0.20 or unrealized_return <= -0.20:
            holding = False
            sell_date = date
            sell_price = current_price
            
            # 判斷是停利還是停損，用於記錄
            exit_reason = '停利 (+20%)' if unrealized_return >= 0.20 else '停損 (-20%)'
            
            trades.append({
                'Buy_Date': buy_date.strftime('%Y-%m-%d'),
                'Buy_Price': round(buy_price, 2),
                'Sell_Date': sell_date.strftime('%Y-%m-%d'),
                'Sell_Price': round(sell_price, 2),
                'Return_Pct': round(unrealized_return * 100, 2),
                'Holding_Days': (sell_date - buy_date).days,
                'Exit_Reason': exit_reason
            })

# 處理最後一天未平倉狀況 (強制結算)
if holding:
    last_date = df.index[-1]
    sell_price = df.iloc[-1]['Close']
    unrealized_return = (sell_price - buy_price) / buy_price
    trades.append({
        'Buy_Date': buy_date.strftime('%Y-%m-%d'),
        'Buy_Price': round(buy_price, 2),
        'Sell_Date': last_date.strftime('%Y-%m-%d'),
        'Sell_Price': round(sell_price, 2),
        'Return_Pct': round(unrealized_return * 100, 2),
        'Holding_Days': (last_date - buy_date).days,
        'Exit_Reason': '回測結束 (未平倉強制結算)'
    })

# ==========================================
# 5. 輸出 DataFrame 與績效總結
# ==========================================
trades_df = pd.DataFrame(trades)

print("\n" + "="*50)
if trades_df.empty:
    print("在此期間內，沒有任何一天同時滿足所有嚴格進場條件。")
else:
    total_trades = len(trades_df)
    win_trades = len(trades_df[trades_df['Return_Pct'] > 0])
    win_rate = (win_trades / total_trades) * 100 if total_trades > 0 else 0
    avg_return = trades_df['Return_Pct'].mean()
    
    # 計算停利與停損的次數
    tp_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停利')])
    sl_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停損')])
    
    print(f"回測完成！總交易次數: {total_trades} 次")
    print(f"成功停利次數: {tp_count} 次 | 觸發停損次數: {sl_count} 次")
    print(f"策略勝率: {win_rate:.2f}%")
    print(f"單筆平均報酬率: {avg_return:.2f}%")
    print("="*50 + "\n")
    
    # 顯示交易明細 (在 Jupyter 裡可以改用 display(trades_df) 會有更漂亮的表格)
    print(trades_df.to_string(index=False))

正在獲取台積電(2330.TW)歷史資料...

回測完成！總交易次數: 2 次
成功停利次數: 1 次 | 觸發停損次數: 1 次
策略勝率: 50.00%
單筆平均報酬率: -0.34%

  Buy_Date  Buy_Price  Sell_Date  Sell_Price  Return_Pct  Holding_Days Exit_Reason
2000-04-10      82.17 2000-07-21       65.11      -20.76           102   停損 (-20%)
2009-03-23      50.05 2009-05-27       60.10       20.08            65   停利 (+20%)


# 固定時間出場

In [79]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# 1. 獲取資料 (鎖定原始收盤價)
# ==========================================
print("正在獲取台積電(2330.TW)歷史資料...")
df = yf.download('2330.TW', start='2000-01-01', end='2025-12-31', auto_adjust=False, progress=False)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0] for c in df.columns]

if 'Adj Close' in df.columns:
    df = df.drop(columns=['Adj Close'])
df = df.dropna()

# ==========================================
# 2. 計算技術指標 (全數基於原始收盤價)
# ==========================================
df['Price_Change_Pct'] = df['Close'].pct_change() * 100

df['MA20'] = df['Close'].rolling(window=20).mean()
df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100

df['Min9'] = df['Low'].rolling(window=9).min()
df['Max9'] = df['High'].rolling(window=9).max()
df['RSV'] = (df['Close'] - df['Min9']) / (df['Max9'] - df['Min9']) * 100
df['RSV'] = df['RSV'].fillna(50)

K, D = [50], [50]
for rsv in df['RSV'].values[1:]:
    k_val = K[-1] * (2/3) + rsv * (1/3)
    d_val = D[-1] * (2/3) + k_val * (1/3)
    K.append(k_val)
    D.append(d_val)
df['K'] = K
df['D'] = D

df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean() 
df['OSC'] = df['DIF'] - df['MACD']                      

# ==========================================
# 3. 建立進場訊號
# ==========================================
c1 = df['Price_Change_Pct'].between(-5,5)
c2 = df['K'].between(25, 75)
c3 = df['Bias20'].between(1, 3)
c4 = df['D'].between(25, 75)
c5 = (df['DIF'] > 0) & (df['MACD'] > 0) & (df['OSC'] > 0)
c6 = df['Close'] > df['MA20']
df['Buy_Signal'] = c1 & c2 & c3 & c4 & c5 & c6

# ==========================================
# 4. 回測狀態機 (複合出場條件)
# ==========================================
trades = []
holding = False
buy_date, buy_price = None, 0.0
days_held = 0  

for i in range(len(df)):
    date = df.index[i]
    row = df.iloc[i]
    entry_signal = row['Buy_Signal']
    
    if not holding and entry_signal:
        # 觸發進場
        holding = True
        buy_date = date
        buy_price = row['Close']
        days_held = 0 
        
    elif holding:
        days_held += 1
        current_price = row['Close']
        unrealized_return = (current_price - buy_price) / buy_price
        is_last_day = (i == len(df) - 1)
        
# 定義出場的獨立條件
        cond_take_profit = unrealized_return >= 0.30  # 賺 30% (依您上方程式碼的數值)
        cond_bias_break = row['Bias20'] <= -20         # 【修改】取消固定停損，改為 20日乖離率跌破 -5% 時出場
        
        # 任一條件觸發，或到達資料最後一天
        if cond_take_profit or cond_bias_break or is_last_day:
            sell_date = date
            sell_price = current_price
            return_pct = unrealized_return
            
            # 判斷是哪個條件觸發的
            if cond_take_profit:
                exit_reason = '停利 (+30%)'
            elif cond_bias_break:
                exit_reason = '乖離率跌破 -20%'
            else:
                exit_reason = f'資料結束強制平倉 (持有 {days_held} 天)'
            
            trades.append({
                'Buy_Date': buy_date.strftime('%Y-%m-%d'),
                'Buy_Price': round(buy_price, 2),
                'Sell_Date': sell_date.strftime('%Y-%m-%d'),
                'Sell_Price': round(sell_price, 2),
                'Return_Pct': round(return_pct * 100, 2),
                'Holding_Days': days_held,
                'Exit_Reason': exit_reason
            })
            
            # 賣出後恢復空手狀態
            holding = False

# ==========================================
# 5. 輸出 DataFrame 與績效總結
# ==========================================
trades_df = pd.DataFrame(trades)

print("\n" + "="*50)
if trades_df.empty:
    print("在此期間內，沒有任何一天同時滿足所有嚴格進場條件。")
else:
    total_trades = len(trades_df)
    win_trades = len(trades_df[trades_df['Return_Pct'] > 0])
    win_rate = (win_trades / total_trades) * 100 if total_trades > 0 else 0
    avg_return = trades_df['Return_Pct'].mean()
    
    # 計算各出場原因的次數
    tp_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停利')])
    sl_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停損')])
    ma_break_count = len(trades_df[trades_df['Exit_Reason'].str.contains('跌破20日均線')])
    
    print(f"回測完成！總交易次數: {total_trades} 次")
    print(f"策略勝率: {win_rate:.2f}%")
    print(f"單筆平均報酬率: {avg_return:.2f}%")
    print("-" * 50)
    print(f"【出場分析】停利: {tp_count} 次 | 停損: {sl_count} 次 | 跌破均線10%: {ma_break_count} 次")
    print("="*50 + "\n")
    
    # 顯示交易明細
    print(trades_df.to_string(index=False))

正在獲取台積電(2330.TW)歷史資料...

回測完成！總交易次數: 15 次
策略勝率: 86.67%
單筆平均報酬率: 18.25%
--------------------------------------------------
【出場分析】停利: 12 次 | 停損: 0 次 | 跌破均線10%: 0 次

  Buy_Date  Buy_Price  Sell_Date  Sell_Price  Return_Pct  Holding_Days       Exit_Reason
2000-04-06      79.04 2001-09-25       33.24      -57.95           383        乖離率跌破 -20%
2002-04-01      66.96 2002-07-29       38.95      -41.83            85        乖離率跌破 -20%
2003-01-21      36.71 2003-07-04       47.82       30.25           118         停利 (+30%)
2003-08-04      46.28 2005-12-21       60.25       30.20           622         停利 (+30%)
2006-01-13      61.69 2012-02-29       81.10       31.47          1513         停利 (+30%)
2012-03-01      80.10 2013-02-06      105.00       31.09           239         停利 (+30%)
2013-05-17     113.50 2015-02-11      148.00       30.40           433         停利 (+30%)
2015-03-03     150.50 2017-05-02      196.50       30.56           530         停利 (+30%)
2017-05-18     203.50 2018-01-23    

# 0050

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# 0050 成分股字典 (標準化代號)
# ==========================================
stock_0050 = {
    '2330.TW': '台積電', '2308.TW': '台達電', '2317.TW': '鴻海', '2454.TW': '聯發科', '3711.TW': '日月光投控',
    '2891.TW': '中信金', '2345.TW': '智邦', '2383.TW': '台光電', '2382.TW': '廣達', '2881.TW': '富邦金',
    '2882.TW': '國泰金', '2303.TW': '聯電', '3017.TW': '奇鋐', '2360.TW': '致茂', '2887.TW': '台新新光金',
    '2412.TW': '中華電', '2884.TW': '玉山金', '2885.TW': '元大金', '2886.TW': '兆豐金', '2890.TW': '永豐金',
    '2357.TW': '華碩', '3231.TW': '緯創', '2327.TW': '國巨', '1303.TW': '南亞', '1216.TW': '統一',
    '6669.TW': '緯穎', '3653.TW': '健策', '2880.TW': '華南金', '2892.TW': '第一金', '2883.TW': '凱基金',
    '2368.TW': '金像電', '2449.TW': '京元電子', '2344.TW': '華邦電', '2301.TW': '光寶科', '5880.TW': '合庫金',
    '2408.TW': '南亞科', '2603.TW': '長榮', '2002.TW': '中鋼', '3008.TW': '大立光', '3661.TW': '世芯-KY',
    '7769.TW': '鴻勁', '1301.TW': '台塑', '2059.TW': '川湖', '4904.TW': '遠傳', '3045.TW': '台灣大',
    '2395.TW': '研華', '2207.TW': '和泰車', '6919.TW': '康霈', '6505.TW': '台塑化'
}

# ==========================================
# 核心回測函數打包
# ==========================================
def backtest_strategy(ticker, stock_name, start_date='2000-01-01', end_date='2025-12-31'):
    """對單一檔股票執行回測，回傳績效字典與交易紀錄 DataFrame"""
    try:
        # 1. 獲取資料
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)
        
        if df.empty:
            return None, pd.DataFrame()
            
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]

        if 'Adj Close' in df.columns:
            df = df.drop(columns=['Adj Close'])
        df = df.dropna()

        if len(df) < 50: # 資料太少不予計算
            return None, pd.DataFrame()

        # 2. 計算技術指標
        df['Price_Change_Pct'] = df['Close'].pct_change() * 100
        
        # 【新增】5 日均線與乖離率 (用於進場條件3)
        df['MA5'] = df['Close'].rolling(window=5).mean()
        df['Bias5'] = (df['Close'] - df['MA5']) / df['MA5'] * 100
        
        # 20 日均線與乖離率 (用於進場條件5 與 出場條件)
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100
        
        df['Min9'] = df['Low'].rolling(window=9).min()
        df['Max9'] = df['High'].rolling(window=9).max()
        df['RSV'] = (df['Close'] - df['Min9']) / (df['Max9'] - df['Min9']) * 100
        df['RSV'] = df['RSV'].fillna(50)

        K, D = [50], [50]
        for rsv in df['RSV'].values[1:]:
            k_val = K[-1] * (2/3) + rsv * (1/3)
            d_val = D[-1] * (2/3) + k_val * (1/3)
            K.append(k_val)
            D.append(d_val)
        df['K'] = K
        df['D'] = D

        df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
        df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
        df['DIF'] = df['EMA12'] - df['EMA26']
        df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean() 
        df['OSC'] = df['DIF'] - df['MACD']                      

        # 3. 建立進場訊號 (依據您的最新條件)
        c1 = df['Price_Change_Pct'].between(4, 10)       # 條件1: 漲幅4%~10%
        c2 = df['K'].between(50, 75)                     # 條件2: K值在50~75
        c3 = df['Bias5'].between(1, 5)                   # 條件3: 5日均線乖離率1%~5%
        c4 = df['D'].between(50, 75)                     # 條件2: D值在50~75
        c5 = (df['DIF'] > 0) & (df['MACD'] > 0) & (df['OSC'] > 0) # 條件4: MACD三指標大於0
        c6 = df['Close'] > df['MA20']                    # 條件5: 成交價在20日均線以上
        
        df['Buy_Signal'] = c1 & c2 & c3 & c4 & c5 & c6

        # 4. 回測狀態機
        trades = []
        holding = False
        buy_date, buy_price = None, 0.0
        days_held = 0  

        for i in range(len(df)):
            date = df.index[i]
            row = df.iloc[i]
            entry_signal = row['Buy_Signal']
            
            if not holding and entry_signal:
                holding = True
                buy_date = date
                buy_price = row['Close']
                days_held = 0 
                
            elif holding:
                days_held += 1
                current_price = row['Close']
                unrealized_return = (current_price - buy_price) / buy_price
                is_last_day = (i == len(df) - 1)
                
                # 維持上一次設定的複合出場條件
                cond_take_profit = unrealized_return >= 0.40 
                cond_bias_break = row['Bias20'] <= -20      
                
                if cond_take_profit or cond_bias_break or is_last_day:
                    sell_date = date
                    sell_price = current_price
                    return_pct = unrealized_return
                    
                    if cond_take_profit:
                        exit_reason = '停利 (+40%)'
                    elif cond_bias_break:
                        exit_reason = '乖離率跌破 -20%'
                    else:
                        exit_reason = f'資料結束強制平倉 (持有 {days_held} 天)'
                    
                    trades.append({
                        'Ticker': ticker,
                        'Name': stock_name,
                        'Buy_Date': buy_date.strftime('%Y-%m-%d'),
                        'Buy_Price': round(buy_price, 2),
                        'Sell_Date': sell_date.strftime('%Y-%m-%d'),
                        'Sell_Price': round(sell_price, 2),
                        'Return_Pct': round(return_pct * 100, 2),
                        'Holding_Days': days_held,
                        'Exit_Reason': exit_reason
                    })
                    holding = False

        trades_df = pd.DataFrame(trades)
        
        # 彙整績效
        if trades_df.empty:
            return None, pd.DataFrame()
            
        total_trades = len(trades_df)
        win_trades = len(trades_df[trades_df['Return_Pct'] > 0])
        win_rate = (win_trades / total_trades) * 100
        avg_return = trades_df['Return_Pct'].mean()
        
        tp_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停利')])
        sl_count = len(trades_df[trades_df['Exit_Reason'].str.contains('乖離率跌破')]) 
        
        summary = {
            '代號': ticker.replace('.TW', ''),
            '名稱': stock_name,
            '交易次數': total_trades,
            '勝率(%)': round(win_rate, 2),
            '平均報酬(%)': round(avg_return, 2),
            '停利次數': tp_count,
            '停損次數': sl_count
        }
        return summary, trades_df
        
    except Exception as e:
        print(f"處理 {ticker} 時發生錯誤: {e}")
        return None, pd.DataFrame()

# ==========================================
# 主程式：批次執行並彙總結果
# ==========================================
if __name__ == "__main__":
    all_summaries = []
    all_trades = pd.DataFrame()
    
    print(f"開始回測 0050 成分股，共 {len(stock_0050)} 檔...")
    
    for ticker, name in stock_0050.items():
        print(f"正在分析: {ticker} {name}")
        summary, trades_df = backtest_strategy(ticker, name)
        
        if summary:
            all_summaries.append(summary)
            all_trades = pd.concat([all_trades, trades_df], ignore_index=True)

    print("\n" + "="*60)
    print(" 0050 成分股回測績效總表 ")
    print("="*60)
    
    if all_summaries:
        summary_df = pd.DataFrame(all_summaries)
        summary_df = summary_df.sort_values(by='勝率(%)', ascending=False).reset_index(drop=True)
        print(summary_df.to_string(index=False))
        
        print("\n[統計]")
        print(f"共有 {len(summary_df)} 檔股票觸發交易訊號。")
        print(f"所有股票平均勝率: {summary_df['勝率(%)'].mean():.2f}%")
        print(f"所有股票單筆平均報酬: {summary_df['平均報酬(%)'].mean():.2f}%")
        
        # 匯出資料
        # all_trades.to_excel('0050_Backtest_Trades.xlsx', index=False)
        # summary_df.to_excel('0050_Backtest_Summary.xlsx', index=False)
    else:
        print("沒有任何股票觸發交易訊號。")

開始回測 0050 成分股，共 49 檔...
正在分析: 2330.TW 台積電
正在分析: 2308.TW 台達電
正在分析: 2317.TW 鴻海
正在分析: 2454.TW 聯發科
正在分析: 3711.TW 日月光投控
正在分析: 2891.TW 中信金
正在分析: 2345.TW 智邦
正在分析: 2383.TW 台光電
正在分析: 2382.TW 廣達
正在分析: 2881.TW 富邦金
正在分析: 2882.TW 國泰金
正在分析: 2303.TW 聯電
正在分析: 3017.TW 奇鋐
正在分析: 2360.TW 致茂
正在分析: 2887.TW 台新新光金
正在分析: 2412.TW 中華電
正在分析: 2884.TW 玉山金
正在分析: 2885.TW 元大金
正在分析: 2886.TW 兆豐金
正在分析: 2890.TW 永豐金
正在分析: 2357.TW 華碩
正在分析: 3231.TW 緯創
正在分析: 2327.TW 國巨
正在分析: 1303.TW 南亞
正在分析: 1216.TW 統一
正在分析: 6669.TW 緯穎
正在分析: 3653.TW 健策
正在分析: 2880.TW 華南金
正在分析: 2892.TW 第一金
正在分析: 2883.TW 凱基金
正在分析: 2368.TW 金像電
正在分析: 2449.TW 京元電子
正在分析: 2344.TW 華邦電
正在分析: 2301.TW 光寶科
正在分析: 5880.TW 合庫金
正在分析: 2408.TW 南亞科
正在分析: 2603.TW 長榮
正在分析: 2002.TW 中鋼
正在分析: 3008.TW 大立光
正在分析: 3661.TW 世芯-KY
正在分析: 7769.TW 鴻勁
正在分析: 1301.TW 台塑
正在分析: 2059.TW 川湖
正在分析: 4904.TW 遠傳
正在分析: 3045.TW 台灣大
正在分析: 2395.TW 研華
正在分析: 2207.TW 和泰車
正在分析: 6919.TW 康霈
正在分析: 6505.TW 台塑化

 0050 成分股回測績效總表 
  代號    名稱  交易次數  勝率(%)  平均報酬(%)  停利次數  停損次數
2892   第一金     2 100.00    44.32     2     0


# 個別股票

In [7]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# 0050 成分股字典 (標準化代號)
# ==========================================
stock_0050 =  {
    '5434.TW': '崇越', '2472.TW': '立隆電', '3004.TW': '豐達科', '2355.TW': '敬鵬', '3033.TW': '威健',
    '2327.TW': '國巨', '3211.TWO': '順達', '3078.TWO': '僑威', '6244.TWO': '茂迪', '1519.TW': '華城',
    '1513.TW': '中興電', '1514.TW': '亞力', '1609.TW': '大亞', '6239.TW': '力成', '6291.TWO': '沛亨',
    '6127.TWO': '九豪', '4939.TWO': '亞電', '6190.TWO': '萬泰科', '1605.TW': '華新', '2481.TW': '強茂',
    '6411.TWO': '晶焱', '5425.TWO': '台半', '2365.TW': '昆盈', '3706.TW': '神達', '4510.TWO': '高峰',
    '1504.TW': '東元', '2455.TW': '全新'
}
# ==========================================
# 核心回測函數打包
# ==========================================
def backtest_strategy(ticker, stock_name, start_date='2000-01-01', end_date='2025-12-31'):
    """對單一檔股票執行回測，回傳績效字典與交易紀錄 DataFrame"""
    try:
        # 1. 獲取股價資料
        df = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False, progress=False)
        
        if df.empty:
            return None, pd.DataFrame()
            
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]

        if 'Adj Close' in df.columns:
            df = df.drop(columns=['Adj Close'])
        df = df.dropna()

        if len(df) < 60: # 確保資料長度足以計算 60 日均線
            return None, pd.DataFrame()

        # 2. 獲取流通股數 (用於計算周轉率)
        # 註: 取用當前流通股數來推算歷史周轉率為近似值
        ticker_obj = yf.Ticker(ticker)
        shares_out = ticker_obj.info.get('sharesOutstanding', None)

        # 3. 計算技術指標
        # (1) 日均線與乖離率
        df['MA20'] = df['Close'].rolling(window=20).mean()
        df['MA60'] = df['Close'].rolling(window=60).mean()
        df['Bias20'] = (df['Close'] - df['MA20']) / df['MA20'] * 100 # 保留給出場條件用
        
        # (2) 日周轉率 (%)
        if shares_out and shares_out > 0:
            df['Turnover_Rate'] = (df['Volume'] / shares_out) * 100
        else:
            df['Turnover_Rate'] = 0 # 若 API 抓不到股數則設為 0

        # (3) 周 MACD (嚴謹計算，避免 Look-ahead bias)
        # 將資料依據「周」進行分組，取得每周最後一個交易日的收盤價
        df['Week_Period'] = df.index.to_period('W')
        df_weekly = df.groupby('Week_Period')['Close'].last().to_frame()
        
        # 算出純粹的周線 MACD
        df_weekly['EMA12_W'] = df_weekly['Close'].ewm(span=12, adjust=False).mean()
        df_weekly['EMA26_W'] = df_weekly['Close'].ewm(span=26, adjust=False).mean()
        df_weekly['DIF_W'] = df_weekly['EMA12_W'] - df_weekly['EMA26_W']
        df_weekly['MACD_W'] = df_weekly['DIF_W'].ewm(span=9, adjust=False).mean()
        df_weekly['OSC_W'] = df_weekly['DIF_W'] - df_weekly['MACD_W']
        
        # 【關鍵步驟】將周指標向下平移一周 (shift 1)。
        # 這樣一來，「本周」的所有日線，都會去讀取「上周」已確認的周 MACD，絕對不會用到未來數據！
        df_weekly[['DIF_W', 'MACD_W', 'OSC_W']] = df_weekly[['DIF_W', 'MACD_W', 'OSC_W']].shift(1)
        
        # 將周 MACD 的數值映射回日線 DataFrame 中
        df = df.join(df_weekly[['DIF_W', 'MACD_W', 'OSC_W']], on='Week_Period')

        # 4. 建立進場訊號 (根據您的自訂篩選圖片)
        # 條件1: 周轉率 2% ~ 10%
        c1 = df['Turnover_Rate'].between(2, 10)
        # 條件2: 周 MACD 落點：DIF、MACD、OSC 皆大於 0
        c2 = (df['DIF_W'] > 0) & (df['MACD_W'] > 0) & (df['OSC_W'] > 0)
        # 特定條件1: 長期趨勢 (20日均線 > 60日均線)
        c3 = df['MA20'] > df['MA60']
        # 特定條件2: 成交價在 60 日均線之上
        c4 = df['Close'] > df['MA60']
        
        df['Buy_Signal'] = c1 & c2 & c3 & c4

        # 5. 回測狀態機 (維持原有的出場邏輯)
        trades = []
        holding = False
        buy_date, buy_price = None, 0.0
        days_held = 0  

        for i in range(len(df)):
            date = df.index[i]
            row = df.iloc[i]
            entry_signal = row['Buy_Signal']
            
            if not holding and entry_signal:
                holding = True
                buy_date = date
                buy_price = row['Close']
                days_held = 0 
                
            elif holding:
                days_held += 1
                current_price = row['Close']
                unrealized_return = (current_price - buy_price) / buy_price
                is_last_day = (i == len(df) - 1)
                
                # 出場條件: 獲利 30% 或 乖離率跌破 -20%
                cond_take_profit = unrealized_return >= 0.30 
                cond_bias_break = row['Bias20'] <= -20       
                
                if cond_take_profit or cond_bias_break or is_last_day:
                    sell_date = date
                    sell_price = current_price
                    return_pct = unrealized_return
                    
                    if cond_take_profit:
                        exit_reason = '停利 (+30%)'
                    elif cond_bias_break:
                        exit_reason = '乖離率跌破 -20%'
                    else:
                        exit_reason = f'資料結束強制平倉 (持有 {days_held} 天)'
                    
                    trades.append({
                        'Ticker': ticker,
                        'Name': stock_name,
                        'Buy_Date': buy_date.strftime('%Y-%m-%d'),
                        'Buy_Price': round(buy_price, 2),
                        'Sell_Date': sell_date.strftime('%Y-%m-%d'),
                        'Sell_Price': round(sell_price, 2),
                        'Return_Pct': round(return_pct * 100, 2),
                        'Holding_Days': days_held,
                        'Exit_Reason': exit_reason
                    })
                    holding = False

        trades_df = pd.DataFrame(trades)
        
        # 彙整績效
        if trades_df.empty:
            return None, pd.DataFrame()
            
        total_trades = len(trades_df)
        win_trades = len(trades_df[trades_df['Return_Pct'] > 0])
        win_rate = (win_trades / total_trades) * 100
        avg_return = trades_df['Return_Pct'].mean()
        
        tp_count = len(trades_df[trades_df['Exit_Reason'].str.contains('停利')])
        sl_count = len(trades_df[trades_df['Exit_Reason'].str.contains('乖離率跌破')]) 
        
        summary = {
            '代號': ticker.replace('.TW', ''),
            '名稱': stock_name,
            '交易次數': total_trades,
            '勝率(%)': round(win_rate, 2),
            '平均報酬(%)': round(avg_return, 2),
            '停利次數': tp_count,
            '停損次數': sl_count
        }
        return summary, trades_df
        
    except Exception as e:
        print(f"處理 {ticker} 時發生錯誤: {e}")
        return None, pd.DataFrame()

# ==========================================
# 主程式：批次執行並彙總結果
# ==========================================
if __name__ == "__main__":
    all_summaries = []
    all_trades = pd.DataFrame()
    
    print(f"開始回測 0050 成分股，共 {len(stock_0050)} 檔...")
    
    # 迴圈跑過每一檔股票
    for ticker, name in stock_0050.items():
        print(f"正在分析: {ticker} {name}")
        summary, trades_df = backtest_strategy(ticker, name)
        
        if summary:
            all_summaries.append(summary)
            all_trades = pd.concat([all_trades, trades_df], ignore_index=True)

    # 輸出最終總表
    print("\n" + "="*60)
    print(" 0050 成分股回測績效總表 ")
    print("="*60)
    
    if all_summaries:
        summary_df = pd.DataFrame(all_summaries)
        # 依照勝率排序
        summary_df = summary_df.sort_values(by='勝率(%)', ascending=False).reset_index(drop=True)
        print(summary_df.to_string(index=False))
        
        print("\n[統計]")
        print(f"共有 {len(summary_df)} 檔股票觸發交易訊號。")
        print(f"所有股票平均勝率: {summary_df['勝率(%)'].mean():.2f}%")
        print(f"所有股票單筆平均報酬: {summary_df['平均報酬(%)'].mean():.2f}%")
        
        # 您可以選擇將詳細交易紀錄匯出成 Excel
        # all_trades.to_excel('0050_Backtest_Trades.xlsx', index=False)
        # summary_df.to_excel('0050_Backtest_Summary.xlsx', index=False)
    else:
        print("沒有任何股票觸發交易訊號。")

開始回測 0050 成分股，共 27 檔...
正在分析: 5434.TW 崇越
正在分析: 2472.TW 立隆電
正在分析: 3004.TW 豐達科
正在分析: 2355.TW 敬鵬
正在分析: 3033.TW 威健
正在分析: 2327.TW 國巨
正在分析: 3211.TWO 順達
正在分析: 3078.TWO 僑威
正在分析: 6244.TWO 茂迪
正在分析: 1519.TW 華城
正在分析: 1513.TW 中興電
正在分析: 1514.TW 亞力
正在分析: 1609.TW 大亞
正在分析: 6239.TW 力成
正在分析: 6291.TWO 沛亨
正在分析: 6127.TWO 九豪
正在分析: 4939.TWO 亞電
正在分析: 6190.TWO 萬泰科
正在分析: 1605.TW 華新
正在分析: 2481.TW 強茂
正在分析: 6411.TWO 晶焱
正在分析: 5425.TWO 台半
正在分析: 2365.TW 昆盈
正在分析: 3706.TW 神達
正在分析: 4510.TWO 高峰
正在分析: 1504.TW 東元
正在分析: 2455.TW 全新

 0050 成分股回測績效總表 
   代號  名稱  交易次數  勝率(%)  平均報酬(%)  停利次數  停損次數
 6239  力成    10  90.00    23.43     8     1
 1519  華城    25  76.00    14.21    18     6
 1504  東元     8  75.00    17.00     6     1
4510O  高峰    12  75.00    15.95     9     2
 1513 中興電    16  75.00    17.55    12     3
 2472 立隆電    18  72.22    11.58    12     5
 1514  亞力    14  71.43    17.31    10     4
 1609  大亞    17  70.59    12.33    12     4
 1605  華新    10  70.00     8.55     6     3
 2327  國巨    20  65.00     7.94    12     7
6